# Vocabulary compression changes the computation

Chapter 2 · Day 2

Why can a larger vocabulary shorten a sequence but increase output-head cost? Use the same text under different merge budgets, then keep parameter storage, per-position output work and attention interactions separate.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(909)
plt.rcParams.update({'figure.figsize': (8, 3.6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

In [ ]:
from dongxi_llms.course_foundations import ByteBPE
text = '数据库 数据库 database database språkmodellen språkmodellen'
corpus = [text] * 8
budgets = [0, 8, 24, 48]
models = [ByteBPE.train(corpus, n) for n in budgets]
sizes = [len(m.pieces) for m in models]
lengths = [len(m.encode(text)) for m in models]
for m in models: assert m.decode(m.encode(text)) == text
print(list(zip(budgets, sizes, lengths)))

Prediction: does fewer tokens imply lower total model cost in every setting? Reference: token compression reduces T for this text. A larger V increases the tied embedding table V×D and a dense output projection's per-position work. An architectural comparison must consider both. The tokenizer does not make a compound equivalent to all of its parts in embedding geometry.

In [ ]:
width = 64
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].plot(sizes, lengths, marker='o')
axes[0].set(xlabel='Toy vocabulary entries V', ylabel='Text length T (tokens)')
axes[1].plot(sizes, [v*width for v in sizes], marker='o')
axes[1].set(xlabel='V', ylabel='Tied table parameters V×D')
axes[2].plot(sizes, [t*t for t in lengths], marker='o')
axes[2].set(xlabel='V', ylabel='Dense attention pair count T²')
fig.suptitle('Measured token lengths; calculated structural costs (D=64)')
fig.tight_layout()
plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 vocabulary and sequence cost: reference plot 01](../figures/chapter-02/day-02-02_vocabulary_and_sequence_cost-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

Reference explanation: the first panel measures this tokenizer fixture. The others calculate structural terms, not GPU latency or a full FLOP count. Causal attention has T(T+1)/2 permitted pairs, while many dense implementations form a T×T score array; neither formula describes every modern attention kernel's memory allocation.

Intervention: train on English only and re-evaluate the same mixed text. Reference: vocabulary growth may compress the frequent training language while leaving Chinese as byte fragments. A single multilingual string does not support a ranking of entire languages or model quality.

Integration checkpoint: could reducing V accidentally make one story exceed a fixed 1024-token context? Reference: yes, if the tokenizer emits more pieces. Context is measured in the model's tokens, not words or characters; changing tokenization changes how much text the same T holds.